## Aggregarte Embeddings

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_parquet("jina_v3_offtheshelf_full.parquet")
data.head(10)

,id,politicianId,party,year,year_month,n_tokens,embedding
0,604683,11001434,SPD,2000,2000-01,1213,"[0.038507473, -0.109700434, 0.03194454, -0.015..."
1,604883,11002754,CDU/CSU,2000,2000-01,1002,"[-0.0038278345, -0.0912512, 0.052730158, -0.03..."
2,604885,11000431,SPD,2000,2000-01,1090,"[-0.0045266734, -0.10698758, 0.043889068, -0.0..."
3,604905,11001180,FDP,2000,2000-01,1560,"[0.045944948, -0.10603965, 0.010365211, -0.052..."
4,605067,11003216,CDU/CSU,2000,2000-01,1559,"[-0.014769425, -0.13127752, 0.04492379, -0.009..."
5,605567,11002457,SPD,2000,2000-01,1997,"[-0.0083636325, -0.16840981, 0.02602508, 0.003..."
6,605569,11002651,CDU/CSU,2000,2000-01,1032,"[-0.08398846, -0.015895868, -0.0641574, 0.0553..."
7,605685,11001033,CDU/CSU,2000,2000-01,1480,"[0.008954962, -0.14312434, 0.030577222, 0.0018..."
8,605717,11002720,Grüne,2000,2000-01,1566,"[0.015453605, -0.07390311, 0.014936259, -0.020..."
9,606154,11003195,Grüne,2000,2000-02,2442,"[0.045142677, -0.0244334, 0.008730959, -0.0239..."


We decided to aggretate by MP and year, as the average number of spreeches per year is 7. 

Let's first check how many MPs gave speeches for different parties within the same year.

In [3]:
party_switches = (
    data.groupby(["politicianId", "year"])["party"]
    .nunique()
    .reset_index(name="n_parties")
)
party_switches = party_switches[party_switches["n_parties"] > 1]

print(party_switches)

      politicianId  year  n_parties
0               -1  2000          3
1               -1  2001          2
2               -1  2002          4
3               -1  2003          2
4               -1  2004          2
5               -1  2005          2
6               -1  2006          2
7               -1  2007          2
8               -1  2008          2
9               -1  2009          2
10              -1  2010          2
13              -1  2013          3
18              -1  2018          2
19              -1  2019          2
20              -1  2020          4
21              -1  2021          4
3109      11002813  2009          2
4340      11003206  2002          2
4343      11003206  2005          2


As this number is very low, we can neglect those cases and simply assign the party they belonged to when they gave their first speech in that year.

In [ ]:
agg_records = []

for (politicianId, year), idx in data.groupby(["politicianId", "year"]).groups.items():
    mean_embedding = np.mean(np.stack(data.loc[idx, "embedding"]), axis=0)
    agg_records.append({
        "politicianId": politicianId,
        "year": year,
        "party": data.loc[idx, "party"].iloc[0],
        "n_speeches": len(idx),
        "embedding": mean_embedding
    })

df_agg = pd.DataFrame(agg_records)

In [6]:
df_agg

,politicianId,year,party,n_speeches,embedding
0,-1,2000,PDS,27,"[0.056539893, -0.09861565, 0.03063002, 0.00181..."
1,-1,2001,PDS,25,"[0.021111745, -0.1159311, 0.028263073, 0.01033..."
2,-1,2002,CDU/CSU,20,"[0.056148797, -0.085535474, 0.052539904, 0.018..."
3,-1,2003,Grüne,38,"[0.057862733, -0.10331881, 0.066418335, 0.0417..."
4,-1,2004,FDP,47,"[0.05605023, -0.09303291, 0.056246977, 0.03644..."
...,...,...,...,...,...
12730,11004949,2020,SPD,4,"[-0.0029243752, -0.110075995, 0.062244855, -0...."
12731,11004949,2021,SPD,3,"[0.0056287777, -0.11775065, 0.088844776, -0.02..."
12732,11004951,2021,Grüne,1,"[0.00035163556, -0.05632198, 0.0993442, 0.0311..."
12733,11004956,2021,FDP,2,"[0.12558612, -0.13013785, 0.041155607, -0.0454..."


In [5]:
df_agg.to_parquet("mp_year_embeddings_agg.parquet")

For comparison with idiology scores (CHES), we need to aggregate by party and year

In [7]:
agg_by_party = []

for (party, year), idx in data.groupby(["party", "year"]).groups.items():
    mean_embedding = np.mean(np.stack(data.loc[idx, "embedding"]), axis=0)
    agg_by_party.append({
        "year": year,
        "party": data.loc[idx, "party"].iloc[0],
        "n_speeches": len(idx),
        "embedding": mean_embedding
    })

df_agg_by_party = pd.DataFrame(agg_by_party)

In [8]:
df_agg_by_party.to_parquet("party_year_embeddings_agg.parquet")